In [ ]:
# 1) Install dependencies, then restart runtime once
import os
import subprocess
import sys
from pathlib import Path

MARKER = Path("/content/.snu_ai_challenge_structured_order_deps_installed")

if not MARKER.exists():
    packages = [
        "transformers>=4.49.0,<4.54.0",
        "accelerate>=0.34.0",
        "bitsandbytes>=0.46.1",
        "peft",
        "qwen-vl-utils",
        "jedi",
        "pandas==2.2.2",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *packages])
    MARKER.write_text("ok")
    print("Dependencies installed. Restarting runtime. Run this cell again after restart.")
    os.kill(os.getpid(), 9)
else:
    print("Dependencies already installed. Continue.")

In [ ]:
# 2) Setup + config
from google.colab import drive
drive.mount("/content/drive")

import ast
import gc
import glob
import itertools
import json
import math
import os
import random
import re
import zipfile
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset, Subset
from transformers import AutoModelForVision2Seq, AutoProcessor, BitsAndBytesConfig, Trainer, TrainingArguments, set_seed
try:
    from transformers import Qwen2VLForConditionalGeneration
except ImportError:
    Qwen2VLForConditionalGeneration = AutoModelForVision2Seq
from peft import PeftModel, prepare_model_for_kbit_training

ZIP_PATH = "/content/drive/MyDrive/SNU_AI_Challenge/snuaichallenge.zip"
DATA_DIR = "/content/snuaichallenge_data"

# Must point to the already trained ORDER LoRA adapter/checkpoint.
ORDER_BEST_CHECKPOINT_DIR = "/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_multitask_v1"
ORDER_BASE_MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"

ORDER_EXPERIMENT_ROOT = "/content/drive/MyDrive/SNU_AI_Challenge/qwen2vl_structured_order_v1"
CACHE_DIR = os.path.join(ORDER_EXPERIMENT_ROOT, "cache")
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_ROOT = os.path.join(ORDER_EXPERIMENT_ROOT, "order_runs", RUN_ID)
EVAL_DIR = os.path.join(RUN_ROOT, "eval")

ORDER_EVENT_CACHE_PATH = os.path.join(CACHE_DIR, "order_sentence_events_v2.json")
ORDER_FRAME_EVIDENCE_CACHE_PATH = os.path.join(CACHE_DIR, "order_frame_evidence_sentence_conditioned_v2.json")

SEED = 42
TRAIN_ROWS = 300
VALID_ROWS = 100
RUN_VARIANTS = ["A_CONTINUE", "D_CONTINUE"]

MAX_TRAIN_STEPS = 50
SAVE_STEPS = 25
LEARNING_RATE = 2e-5
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8

GREEDY_EVAL_ROWS = 100
CANDIDATE_SCORE_EVAL_ROWS = 30
ORDER_CANDIDATE_BATCH_SIZE = 1
ORDER_OUTPUT_WITH_SPACES = True

# Time-budget switches.
RUN_BASELINE_GREEDY_EVAL = True
RUN_BASELINE_24WAY_EVAL = True
GENERATE_STRUCTURED_CACHE = True
RUN_TRAINING = True
RUN_GREEDY_VARIANT_EVAL = True
RUN_WINNER_24WAY_EVAL = True

MIN_PIXELS = 128 * 28 * 28
MAX_PIXELS = 256 * 28 * 28
SMOKE_TEST = False

A_OUTPUT_DIR = os.path.join(RUN_ROOT, "A_continue")
D_OUTPUT_DIR = os.path.join(RUN_ROOT, "D_continue")
VARIANT_TO_OUTPUT_DIR = {"A_CONTINUE": A_OUTPUT_DIR, "D_CONTINUE": D_OUTPUT_DIR}

TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TEST_CSV = os.path.join(DATA_DIR, "test.csv")
TRAIN_IMAGE_DIR = os.path.join(DATA_DIR, "train")
TEST_IMAGE_DIR = os.path.join(DATA_DIR, "test")

if not os.path.isdir(DATA_DIR):
    with zipfile.ZipFile(ZIP_PATH) as zip_file:
        zip_file.extractall("/content/")

for path in [ORDER_EXPERIMENT_ROOT, CACHE_DIR, RUN_ROOT, EVAL_DIR, A_OUTPUT_DIR, D_OUTPUT_DIR]:
    os.makedirs(path, exist_ok=True)

assert os.path.exists(TRAIN_CSV), TRAIN_CSV
assert os.path.isdir(TRAIN_IMAGE_DIR), TRAIN_IMAGE_DIR

def reset_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    set_seed(seed)

reset_all_seeds(SEED)

print("device:", "cuda" if torch.cuda.is_available() else "cpu")
print("run root:", RUN_ROOT)
print("ORDER checkpoint:", ORDER_BEST_CHECKPOINT_DIR)


In [ ]:
# 3) Data split + prompt helpers
def parse_answer(answer):
    result = answer if isinstance(answer, list) else ast.literal_eval(str(answer))
    result = [int(value) for value in result]
    if len(result) != 4 or sorted(result) != [1, 2, 3, 4]:
        raise ValueError(f"Invalid Answer: {answer}")
    return result

def format_answer(answer):
    separator = ", " if ORDER_OUTPUT_WITH_SPACES else ","
    return "[" + separator.join(str(int(value)) for value in answer) + "]"

def parse_prediction(text):
    match = re.fullmatch(
        r"\s*\[\s*([1-4])\s*,\s*([1-4])\s*,\s*([1-4])\s*,\s*([1-4])\s*\]\s*",
        str(text),
    )
    if not match:
        return None
    prediction = [int(value) for value in match.groups()]
    return prediction if sorted(prediction) == [1, 2, 3, 4] else None

def load_rgb(path):
    with Image.open(path) as image:
        return image.convert("RGB").copy()

def load_optional_json(path):
    if not path or not os.path.exists(path):
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json(path, value):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(value, f, ensure_ascii=False, indent=2)

def ensure_adapter_dir(path):
    if not os.path.exists(os.path.join(path, "adapter_config.json")):
        raise RuntimeError(
            "ORDER_BEST_CHECKPOINT_DIR must be a saved LoRA adapter/checkpoint containing adapter_config.json. "
            f"Got: {path}"
        )

train_df = pd.read_csv(TRAIN_CSV)
train_df["Id"] = train_df["Id"].astype(str)
train_df["Answer_list"] = train_df["Answer"].apply(parse_answer)

test_df = pd.read_csv(TEST_CSV)
test_df["Id"] = test_df["Id"].astype(str)

unique_ids = train_df["Id"].unique().copy()
rng = np.random.default_rng(SEED)
rng.shuffle(unique_ids)
valid_size = max(1, int(len(unique_ids) * 0.1))
valid_ids = set(unique_ids[:valid_size])
training_ids = set(unique_ids[valid_size:])

training_df = train_df[train_df["Id"].isin(training_ids)].reset_index(drop=True)
validation_df = train_df[train_df["Id"].isin(valid_ids)].reset_index(drop=True)
experiment_training_df = training_df.sample(n=min(TRAIN_ROWS, len(training_df)), random_state=SEED).reset_index(drop=True)
experiment_validation_df = validation_df.sample(n=min(VALID_ROWS, len(validation_df)), random_state=SEED).reset_index(drop=True)
candidate_eval_df = experiment_validation_df.iloc[:CANDIDATE_SCORE_EVAL_ROWS].copy().reset_index(drop=True)

assert set(training_df["Id"]).isdisjoint(set(validation_df["Id"]))
assert set(experiment_training_df["Id"]).isdisjoint(set(experiment_validation_df["Id"]))
assert candidate_eval_df["Id"].tolist() == experiment_validation_df.iloc[:CANDIDATE_SCORE_EVAL_ROWS]["Id"].tolist()

ORDER_CANDIDATES = list(itertools.permutations([1, 2, 3, 4]))
assert len(ORDER_CANDIDATES) == 24
assert len(set(ORDER_CANDIDATES)) == 24

ORDER_EVENT_CACHE = load_optional_json(ORDER_EVENT_CACHE_PATH)
ORDER_FRAME_EVIDENCE_CACHE = load_optional_json(ORDER_FRAME_EVIDENCE_CACHE_PATH)

def drop_empty(value):
    if isinstance(value, dict):
        cleaned = {key: drop_empty(item) for key, item in value.items()}
        return {key: item for key, item in cleaned.items() if item not in ("", [], {}, None)}
    if isinstance(value, list):
        cleaned = [drop_empty(item) for item in value]
        return [item for item in cleaned if item not in ("", [], {}, None)]
    return value

def compact_json(value):
    return json.dumps(drop_empty(value), ensure_ascii=False, separators=(",", ":"))

def row_image_paths(row, image_root=TRAIN_IMAGE_DIR):
    sample_id = str(row["Id"])
    return [os.path.join(image_root, sample_id, str(row[f"Input_{i}"])) for i in range(1, 5)]

def lookup_events(sample_id):
    value = ORDER_EVENT_CACHE.get(str(sample_id), {})
    return value if isinstance(value, dict) else {"ordering_type": "mixed", "events": value}

def lookup_frame_evidence(sample_id, input_number):
    sample = ORDER_FRAME_EVIDENCE_CACHE.get(str(sample_id), {})
    if not isinstance(sample, dict):
        return {}
    return sample.get(f"Input_{input_number}", {})

def build_order_instruction(sentence, variant, sample_id=None):
    if variant == "A_CONTINUE":
        return (
            f"Sentence:\n{sentence}\n\n"
            "Determine the chronological order of the four images.\n"
            "Output the temporal rank of Input 1, Input 2, Input 3, and Input 4.\n"
            "Output format: [rank1,rank2,rank3,rank4].\n"
            "The output must be a permutation of 1, 2, 3, and 4.\n"
            "Do not output any explanation."
        )
    if variant == "D_CONTINUE":
        evidence_blocks = []
        for input_number in range(1, 5):
            evidence_blocks.append(
                f"Input {input_number} evidence:\n"
                f"{compact_json(lookup_frame_evidence(sample_id, input_number))}"
            )
        return (
            f"Sentence:\n{sentence}\n\n"
            f"Ordered event structure:\n{compact_json(lookup_events(sample_id))}\n\n"
            + "\n\n".join(evidence_blocks)
            + "\n\n"
            "The original images are the primary evidence.\n"
            "Use the structured observations only as supporting evidence.\n"
            "First match each image to the relevant sentence event.\n"
            "For images showing the same event, compare visible progress and state changes.\n"
            "For images showing different events, follow the event order expressed in the sentence.\n"
            "Output the temporal rank of Input 1, Input 2, Input 3, and Input 4.\n"
            "Output format: [rank1,rank2,rank3,rank4].\n"
            "The output must be a permutation of 1, 2, 3, and 4.\n"
            "Do not output any explanation."
        )
    raise ValueError(f"Unknown variant: {variant}")

def row_to_example(row, variant, image_root=TRAIN_IMAGE_DIR, include_answer=True):
    sample_id = str(row["Id"])
    sentence = "" if pd.isna(row["Sentence"]) else str(row["Sentence"])
    example = {
        "Id": sample_id,
        "image_paths": row_image_paths(row, image_root),
        "instruction": build_order_instruction(sentence, variant, sample_id),
        "sentence": sentence,
    }
    if include_answer:
        answer = row["Answer_list"] if "Answer_list" in row else parse_answer(row["Answer"])
        example["answer"] = [int(value) for value in answer]
        example["target_text"] = format_answer(example["answer"])
    return example

def make_order_messages(example):
    content = []
    for input_number in range(1, 5):
        content.append({"type": "text", "text": f"\nInput {input_number}:"})
        content.append({"type": "image"})
    content.append({"type": "text", "text": "\n\n" + example["instruction"]})
    return [{"role": "user", "content": content}]

print("train/valid:", len(training_df), len(validation_df))
print("experiment train/valid:", len(experiment_training_df), len(experiment_validation_df))
ensure_adapter_dir(ORDER_BEST_CHECKPOINT_DIR)


In [ ]:
# 4) Model, collator, greedy evaluation, 24-way scoring
processor = AutoProcessor.from_pretrained(ORDER_BASE_MODEL_ID, min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
processor.tokenizer.padding_side = "right"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

def disable_sampling_warnings(model):
    config = getattr(model, "generation_config", None)
    if config is not None:
        config.do_sample = False
        config.temperature = None
        config.top_p = None
        config.top_k = None

def load_order_adapter(adapter_dir, trainable=False):
    ensure_adapter_dir(adapter_dir)
    base_model = Qwen2VLForConditionalGeneration.from_pretrained(
        ORDER_BASE_MODEL_ID,
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    if trainable:
        base_model.config.use_cache = False
        base_model = prepare_model_for_kbit_training(base_model, use_gradient_checkpointing=True)
    model = PeftModel.from_pretrained(base_model, adapter_dir, is_trainable=trainable)
    model.config.use_cache = not trainable
    disable_sampling_warnings(model)
    return model

def find_last_subsequence(sequence, pattern):
    for start in range(len(sequence) - len(pattern), -1, -1):
        if sequence[start:start + len(pattern)] == pattern:
            return start
    return -1

class OrderDataset(Dataset):
    def __init__(self, dataframe, variant):
        self.dataframe = dataframe.reset_index(drop=True)
        self.variant = variant
    def __len__(self):
        return len(self.dataframe)
    def __getitem__(self, index):
        example = row_to_example(self.dataframe.iloc[index], self.variant, TRAIN_IMAGE_DIR, True)
        example["row_index"] = index
        return example

class QwenOrderCollator:
    def __init__(self, processor):
        self.processor = processor
        self.assistant_prefix_ids = processor.tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)
    def __call__(self, examples):
        if len(examples) != 1:
            raise ValueError("Use batch size 1 with this collator.")
        example = examples[0]
        images = [load_rgb(path) for path in example["image_paths"]]
        messages = make_order_messages(example) + [{"role": "assistant", "content": example["target_text"]}]
        text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        inputs = self.processor(text=[text], images=images, padding=False, return_tensors="pt")
        input_ids = inputs["input_ids"][0].tolist()
        assistant_pos = find_last_subsequence(input_ids, self.assistant_prefix_ids)
        if assistant_pos >= 0:
            answer_start = assistant_pos + len(self.assistant_prefix_ids)
        else:
            prompt_text = self.processor.apply_chat_template(make_order_messages(example), tokenize=False, add_generation_prompt=True)
            prompt_inputs = self.processor(text=[prompt_text], images=images, padding=False, return_tensors="pt")
            answer_start = prompt_inputs["input_ids"].shape[1]
        labels = inputs["input_ids"].clone()
        labels[:, :answer_start] = -100
        if "attention_mask" in inputs:
            labels[inputs["attention_mask"] == 0] = -100
        inputs["labels"] = labels
        return inputs

@torch.no_grad()
def generate_order_prediction(model, example):
    model.eval()
    text = processor.apply_chat_template(make_order_messages(example), tokenize=False, add_generation_prompt=True)
    images = [load_rgb(path) for path in example["image_paths"]]
    inputs = processor(text=[text], images=images, return_tensors="pt").to(model.device)
    generated = model.generate(**inputs, max_new_tokens=16, do_sample=False)
    output_ids = generated[0, inputs.input_ids.shape[1]:]
    output_text = processor.decode(output_ids, skip_special_tokens=True).strip()
    return output_text, parse_prediction(output_text)

def compute_metrics(rows):
    df = pd.DataFrame(rows)
    if df.empty:
        return {"position_accuracy": np.nan, "exact_match_accuracy": np.nan, "valid_output_rate": np.nan, "total": 0}
    return {
        "position_accuracy": float(df["position_correct"].sum() / (len(df) * 4)),
        "exact_match_accuracy": float(df["exact_correct"].mean()),
        "valid_output_rate": float(df["valid_output"].mean()),
        "total": int(len(df)),
    }

@torch.no_grad()
def evaluate_greedy(model, dataframe, variant, output_csv, limit_rows=None, desc="greedy"):
    eval_df = dataframe.iloc[:limit_rows].copy() if limit_rows is not None else dataframe.copy()
    rows = []
    for _, row in tqdm(list(eval_df.iterrows()), total=len(eval_df), desc=desc):
        example = row_to_example(row, variant, TRAIN_IMAGE_DIR, True)
        output_text, prediction = generate_order_prediction(model, example)
        valid = prediction is not None
        answer = example["answer"]
        rows.append({
            "Id": example["Id"],
            "variant": variant,
            "output_text": output_text,
            "Prediction": str(prediction) if prediction else "",
            "Answer": str(answer),
            "valid_output": valid,
            "position_correct": sum(p == a for p, a in zip(prediction, answer)) if valid else 0,
            "exact_correct": int(valid and prediction == answer),
            "No_ordering": bool(row.get("No_ordering", False)) if "No_ordering" in row else None,
            "ordering_type": lookup_events(example["Id"]).get("ordering_type", ""),
        })
    pred_df = pd.DataFrame(rows)
    pred_df.to_csv(output_csv, index=False)
    return pred_df, compute_metrics(rows)

@torch.no_grad()
def score_candidates_for_example(model, example):
    model.eval()
    images = [load_rgb(path) for path in example["image_paths"]]
    assistant_prefix_ids = processor.tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)
    im_end_ids = processor.tokenizer.encode("<|im_end|>", add_special_tokens=False)

    def find_subsequence_from(sequence, pattern, start):
        for index in range(start, len(sequence) - len(pattern) + 1):
            if sequence[index:index + len(pattern)] == pattern:
                return index
        return -1

    rows = []
    for start in range(0, len(ORDER_CANDIDATES), ORDER_CANDIDATE_BATCH_SIZE):
        candidates = ORDER_CANDIDATES[start:start + ORDER_CANDIDATE_BATCH_SIZE]
        candidate_texts = [format_answer(candidate) for candidate in candidates]
        full_texts = []
        for candidate_text in candidate_texts:
            messages = make_order_messages(example) + [{"role": "assistant", "content": candidate_text}]
            full_texts.append(
                processor.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=False,
                )
            )
        inputs = processor(
            text=full_texts,
            images=[images for _ in candidate_texts],
            padding=True,
            return_tensors="pt",
        ).to(model.device)
        outputs = model(**inputs)
        for batch_index, (candidate, candidate_text) in enumerate(zip(candidates, candidate_texts)):
            input_ids = inputs["input_ids"][batch_index].detach().cpu().tolist()
            assistant_pos = find_last_subsequence(input_ids, assistant_prefix_ids)
            if assistant_pos < 0:
                raise RuntimeError("Could not find assistant prefix while scoring ORDER candidates.")
            answer_start = assistant_pos + len(assistant_prefix_ids)
            answer_end = find_subsequence_from(input_ids, im_end_ids, answer_start)
            if answer_end < 0:
                answer_end = answer_start + len(processor.tokenizer.encode(candidate_text, add_special_tokens=False))
            score = 0.0
            used = 0
            for target_pos in range(answer_start, answer_end):
                if target_pos >= inputs["input_ids"].shape[1]:
                    break
                target_id = int(inputs["input_ids"][batch_index, target_pos])
                selected_logits = outputs.logits[batch_index, target_pos - 1, :].float()
                selected_log_probs = torch.log_softmax(selected_logits, dim=-1)
                score += float(selected_log_probs[target_id].detach().cpu())
                used += 1
            rows.append({
                "Id": example["Id"],
                "candidate": str(list(candidate)),
                "candidate_text": candidate_text,
                "sum_log_prob": score,
                "mean_log_prob": score / max(used, 1),
                "candidate_tokens": used,
            })
    rows = sorted(rows, key=lambda item: item["sum_log_prob"], reverse=True)
    for rank, row in enumerate(rows, start=1):
        row["candidate_rank"] = rank
    if len(rows) > 1:
        rows[0]["second_best_score"] = rows[1]["sum_log_prob"]
        rows[0]["score_margin"] = rows[0]["sum_log_prob"] - rows[1]["sum_log_prob"]
    return rows

@torch.no_grad()
def evaluate_24way(model, dataframe, variant, output_prefix, limit_rows=CANDIDATE_SCORE_EVAL_ROWS, desc="24-way"):
    eval_df = dataframe.iloc[:limit_rows].copy().reset_index(drop=True)
    candidate_rows = []
    prediction_rows = []
    for _, row in tqdm(list(eval_df.iterrows()), total=len(eval_df), desc=desc):
        example = row_to_example(row, variant, TRAIN_IMAGE_DIR, True)
        rows = score_candidates_for_example(model, example)
        candidate_rows.extend(rows)
        best = rows[0]
        prediction = ast.literal_eval(best["candidate"])
        answer = example["answer"]
        prediction_rows.append({
            "Id": example["Id"],
            "variant": variant,
            "Prediction": str(prediction),
            "Answer": str(answer),
            "valid_output": True,
            "position_correct": sum(p == a for p, a in zip(prediction, answer)),
            "exact_correct": int(prediction == answer),
            "best_score": best["sum_log_prob"],
            "second_best_score": best.get("second_best_score", np.nan),
            "score_margin": best.get("score_margin", np.nan),
        })
    candidate_df = pd.DataFrame(candidate_rows)
    prediction_df = pd.DataFrame(prediction_rows)
    candidate_df.to_csv(f"{output_prefix}_candidate_scores.csv", index=False)
    prediction_df.to_csv(f"{output_prefix}_24way_predictions.csv", index=False)
    return prediction_df, compute_metrics(prediction_rows), candidate_df

print("model helpers ready")


In [ ]:
# 5) Phase 1: existing ORDER greedy vs 24-way on fixed validation 30
baseline_rows = []
if RUN_BASELINE_GREEDY_EVAL or RUN_BASELINE_24WAY_EVAL:
    model = load_order_adapter(ORDER_BEST_CHECKPOINT_DIR, trainable=False)
    try:
        if RUN_BASELINE_GREEDY_EVAL:
            pred_df, metrics = evaluate_greedy(
                model,
                experiment_validation_df,
                "A_CONTINUE",
                os.path.join(EVAL_DIR, "baseline_order_greedy_predictions.csv"),
                limit_rows=CANDIDATE_SCORE_EVAL_ROWS,
                desc="baseline greedy",
            )
            metrics.update({
                "model": "BASE_ORDER",
                "inference": "greedy",
                "checkpoint": ORDER_BEST_CHECKPOINT_DIR,
                "eval_rows": CANDIDATE_SCORE_EVAL_ROWS,
                "metric_group": "inference_comparison_30",
            })
            baseline_rows.append(metrics)
            print(metrics)
        if RUN_BASELINE_24WAY_EVAL:
            pred_df, metrics, candidate_df = evaluate_24way(
                model,
                candidate_eval_df,
                "A_CONTINUE",
                os.path.join(EVAL_DIR, "baseline_order"),
                limit_rows=CANDIDATE_SCORE_EVAL_ROWS,
                desc="baseline 24-way",
            )
            metrics.update({
                "model": "BASE_ORDER",
                "inference": "24-way",
                "checkpoint": ORDER_BEST_CHECKPOINT_DIR,
                "eval_rows": CANDIDATE_SCORE_EVAL_ROWS,
                "metric_group": "inference_comparison_30",
            })
            baseline_rows.append(metrics)
            print(metrics)
    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()

baseline_metrics_df = pd.DataFrame(baseline_rows)
baseline_metrics_path = os.path.join(EVAL_DIR, "baseline_greedy_vs_24way_metrics.csv")
baseline_metrics_df.to_csv(baseline_metrics_path, index=False)
display(baseline_metrics_df)


In [ ]:
# 6) Generate D_CONTINUE structured cache
def extract_json_object(text):
    match = re.search(r"\{.*\}", str(text), flags=re.DOTALL)
    raw = match.group(0) if match else str(text)
    try:
        return json.loads(raw)
    except Exception:
        return {"raw_text": str(text)}

def normalize_event_structure(value):
    if not isinstance(value, dict):
        value = {"raw_text": str(value)}
    ordering_type = value.get("ordering_type", "mixed")
    if ordering_type not in {"continuous_phase", "distinct_events", "mixed"}:
        ordering_type = "mixed"
    events = value.get("events", [])
    if not isinstance(events, list):
        events = []
    cleaned = []
    for index, event in enumerate(events[:6], start=1):
        if not isinstance(event, dict):
            event = {"action": str(event)}
        cleaned.append({
            "event_id": index,
            "actor": str(event.get("actor", "")),
            "action": str(event.get("action", "")),
            "objects": event.get("objects", []) if isinstance(event.get("objects", []), list) else [],
            "location": str(event.get("location", "")),
            "precondition": str(event.get("precondition", "")),
            "result_state": str(event.get("result_state", "")),
            "temporal_relation": str(event.get("temporal_relation", "")),
        })
    if not cleaned:
        cleaned = [{"event_id": 1, "actor": "", "action": "", "objects": [], "location": "", "precondition": "", "result_state": "", "temporal_relation": ""}]
    return {"ordering_type": ordering_type, "events": cleaned}

def normalize_frame_evidence(value):
    if not isinstance(value, dict):
        value = {"uncertainty": [str(value)]}
    keys = [
        "visible_actors",
        "salient_objects",
        "current_actions",
        "actor_object_interactions",
        "visible_states_or_results",
        "event_matches",
        "continuity_or_transition_cues",
        "uncertainty",
    ]
    out = {"scene_or_location": str(value.get("scene_or_location", ""))}
    for key in keys:
        out[key] = value.get(key, []) if isinstance(value.get(key, []), list) else []
    return out

def cache_rows():
    rows = pd.concat([experiment_training_df, experiment_validation_df], ignore_index=True, sort=False)
    rows["Id"] = rows["Id"].astype(str)
    return rows.drop_duplicates("Id").reset_index(drop=True)

def load_cache_model():
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        ORDER_BASE_MODEL_ID,
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    model.eval()
    model.config.use_cache = True
    disable_sampling_warnings(model)
    return model

@torch.no_grad()
def generate_text_batch(model, messages_batch, images_batch=None, max_new_tokens=220):
    old_padding_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "left"
    try:
        texts = [processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) for messages in messages_batch]
        kwargs = {"text": texts, "padding": True, "return_tensors": "pt"}
        if images_batch is not None:
            kwargs["images"] = images_batch
        inputs = processor(**kwargs).to(model.device)
        generated = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        input_length = inputs["input_ids"].shape[1]
        trimmed = [out_ids[input_length:] for out_ids in generated]
        return processor.batch_decode(trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    finally:
        processor.tokenizer.padding_side = old_padding_side

def chunked(items, chunk_size):
    for start in range(0, len(items), chunk_size):
        yield items[start:start + chunk_size]

def valid_event_cache(value):
    if not isinstance(value, dict):
        return False
    events = value.get("events", [])
    if not events:
        return False
    return any(str(event.get("action", "")).strip() for event in events if isinstance(event, dict))

def valid_frame_cache(value):
    if not isinstance(value, dict):
        return False
    useful_keys = [
        "current_actions",
        "salient_objects",
        "visible_states_or_results",
        "event_matches",
        "scene_or_location",
    ]
    return any(value.get(key) for key in useful_keys)

def validate_cache(rows):
    missing_events = []
    missing_frames = []
    for _, row in rows.iterrows():
        sample_id = str(row["Id"])
        if not valid_event_cache(ORDER_EVENT_CACHE.get(sample_id)):
            missing_events.append(sample_id)
        frames = ORDER_FRAME_EVIDENCE_CACHE.get(sample_id, {})
        for input_number in range(1, 5):
            if not valid_frame_cache(frames.get(f"Input_{input_number}")):
                missing_frames.append(f"{sample_id}:Input_{input_number}")
    if missing_events or missing_frames:
        raise RuntimeError(f"Cache incomplete. missing_events={missing_events[:10]}, missing_frames={missing_frames[:10]}")

if GENERATE_STRUCTURED_CACHE:
    ORDER_EVENT_CACHE = load_optional_json(ORDER_EVENT_CACHE_PATH)
    ORDER_FRAME_EVIDENCE_CACHE = load_optional_json(ORDER_FRAME_EVIDENCE_CACHE_PATH)
    rows = cache_rows()
    cache_model = load_cache_model()
    try:
        event_jobs = []
        for _, row in rows.iterrows():
            sample_id = str(row["Id"])
            if valid_event_cache(ORDER_EVENT_CACHE.get(sample_id)):
                continue
            sentence = "" if pd.isna(row["Sentence"]) else str(row["Sentence"])
            prompt = (
                "Convert the sentence into a compact chronological event structure. "
                "Use only the sentence. Do not use answers, image order, or hidden ground truth. "
                "Do not invent events. Use 1 to 6 events maximum. "
                "Preserve before, after, then, while, finally relations. "
                "Return only valid JSON: "
                "{\"ordering_type\":\"continuous_phase|distinct_events|mixed\","
                "\"events\":[{\"event_id\":1,\"actor\":\"\",\"action\":\"\",\"objects\":[],"
                "\"location\":\"\",\"precondition\":\"\",\"result_state\":\"\",\"temporal_relation\":\"\"}]}\n\n"
                f"Sentence:\n{sentence}"
            )
            event_jobs.append((sample_id, [{"role": "user", "content": [{"type": "text", "text": prompt}]}]))
        event_total = math.ceil(len(event_jobs) / 8) if event_jobs else 0
        for batch_index, batch in enumerate(tqdm(chunked(event_jobs, 8), total=event_total, desc="event cache"), start=1):
            outputs = generate_text_batch(cache_model, [messages for _, messages in batch], max_new_tokens=180)
            for (sample_id, _), output in zip(batch, outputs):
                ORDER_EVENT_CACHE[sample_id] = normalize_event_structure(extract_json_object(output))
            if batch_index % 25 == 0:
                save_json(ORDER_EVENT_CACHE_PATH, ORDER_EVENT_CACHE)
        save_json(ORDER_EVENT_CACHE_PATH, ORDER_EVENT_CACHE)

        frame_jobs = []
        for _, row in rows.iterrows():
            sample_id = str(row["Id"])
            sentence = "" if pd.isna(row["Sentence"]) else str(row["Sentence"])
            if sample_id not in ORDER_FRAME_EVIDENCE_CACHE:
                ORDER_FRAME_EVIDENCE_CACHE[sample_id] = {}
            for input_number in range(1, 5):
                input_key = f"Input_{input_number}"
                if valid_frame_cache(ORDER_FRAME_EVIDENCE_CACHE[sample_id].get(input_key)):
                    continue
                image_path = os.path.join(TRAIN_IMAGE_DIR, sample_id, str(row[input_key]))
                prompt = (
                    "Describe structured visual evidence for this single frame. "
                    "Use the sentence and event structure only to decide what evidence is relevant. "
                    "Do not use other frames, answers, labels, or the true order. "
                    "Prefer event phase, state changes, object continuity, and actor-object interaction. "
                    "Avoid irrelevant background colors, generic clothing/body descriptions, and duplicates. "
                    "Return only valid JSON with keys: scene_or_location, visible_actors, salient_objects, "
                    "current_actions, actor_object_interactions, visible_states_or_results, event_matches, "
                    "continuity_or_transition_cues, uncertainty.\n\n"
                    f"Sentence:\n{sentence}\n\nEvent structure:\n{compact_json(ORDER_EVENT_CACHE[sample_id])}"
                )
                messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
                frame_jobs.append((sample_id, input_key, image_path, messages))
        frame_total = math.ceil(len(frame_jobs) / 2) if frame_jobs else 0
        for batch_index, batch in enumerate(tqdm(chunked(frame_jobs, 2), total=frame_total, desc="frame evidence cache"), start=1):
            outputs = generate_text_batch(
                cache_model,
                [messages for _, _, _, messages in batch],
                images_batch=[[load_rgb(path)] for _, _, path, _ in batch],
                max_new_tokens=220,
            )
            for (sample_id, input_key, _, _), output in zip(batch, outputs):
                ORDER_FRAME_EVIDENCE_CACHE.setdefault(sample_id, {})[input_key] = normalize_frame_evidence(extract_json_object(output))
            if batch_index % 25 == 0:
                save_json(ORDER_FRAME_EVIDENCE_CACHE_PATH, ORDER_FRAME_EVIDENCE_CACHE)
        save_json(ORDER_FRAME_EVIDENCE_CACHE_PATH, ORDER_FRAME_EVIDENCE_CACHE)
    finally:
        del cache_model
        gc.collect()
        torch.cuda.empty_cache()

ORDER_EVENT_CACHE = load_optional_json(ORDER_EVENT_CACHE_PATH)
ORDER_FRAME_EVIDENCE_CACHE = load_optional_json(ORDER_FRAME_EVIDENCE_CACHE_PATH)
validate_cache(cache_rows())
print("event cache:", len(ORDER_EVENT_CACHE))
print("frame evidence cache:", len(ORDER_FRAME_EVIDENCE_CACHE))


In [ ]:
# 7) Debug one D_CONTINUE sample before training
sample = row_to_example(experiment_training_df.iloc[0], "D_CONTINUE", TRAIN_IMAGE_DIR, True)
print("Id:", sample["Id"])
print("image paths exist:", [os.path.exists(path) for path in sample["image_paths"]])
print("target answer:", sample["target_text"])
print("instruction preview:\n", sample["instruction"][:3000])

text = processor.apply_chat_template(make_order_messages(sample), tokenize=False, add_generation_prompt=True)
inputs = processor(text=[text], images=[load_rgb(path) for path in sample["image_paths"]], return_tensors="pt")
decoded = processor.tokenizer.decode(inputs["input_ids"][0], skip_special_tokens=False)
print("has pixel_values:", "pixel_values" in inputs)
print("image_grid_thw:", inputs.get("image_grid_thw"))
print("attention_mask shape:", inputs.get("attention_mask").shape if "attention_mask" in inputs else None)
print("contains Sentence:", "Sentence:" in decoded)
print("contains Ordered event structure:", "Ordered event structure:" in decoded)
print("contains Input 1 evidence:", "Input 1 evidence:" in decoded)
print("answer leaked in prompt:", sample["target_text"] in decoded)

In [ ]:
# 8) Train A_CONTINUE and D_CONTINUE for 50 steps from the same ORDER checkpoint
def train_variant(variant):
    reset_all_seeds(SEED)
    output_dir = VARIANT_TO_OUTPUT_DIR[variant]
    model = load_order_adapter(ORDER_BEST_CHECKPOINT_DIR, trainable=True)
    trainable = [name for name, parameter in model.named_parameters() if parameter.requires_grad]
    print("variant:", variant)
    print("trainable parameter module count:", len(trainable))
    model.print_trainable_parameters()

    train_dataset = OrderDataset(experiment_training_df, variant)
    max_steps = 10 if SMOKE_TEST else MAX_TRAIN_STEPS
    save_steps = max_steps if SMOKE_TEST else SAVE_STEPS

    args = {
        "output_dir": output_dir,
        "num_train_epochs": 1,
        "max_steps": max_steps,
        "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "learning_rate": LEARNING_RATE,
        "warmup_steps": max(1, int(max_steps * 0.05)),
        "max_grad_norm": 0.3,
        "fp16": True,
        "bf16": False,
        "gradient_checkpointing": True,
        "gradient_checkpointing_kwargs": {"use_reentrant": False},
        "optim": "paged_adamw_8bit",
        "logging_steps": 5,
        "save_strategy": "steps",
        "save_steps": save_steps,
        "save_total_limit": None,
        "report_to": "none",
        "remove_unused_columns": False,
        "dataloader_num_workers": 0,
        "seed": SEED,
        "data_seed": SEED,
    }
    if "eval_strategy" in TrainingArguments.__init__.__code__.co_varnames:
        args["eval_strategy"] = "no"
    else:
        args["evaluation_strategy"] = "no"

    trainer = Trainer(
        model=model,
        args=TrainingArguments(**args),
        train_dataset=train_dataset,
        data_collator=QwenOrderCollator(processor),
    )
    trainer.train()
    trainer.save_model(output_dir)
    processor.save_pretrained(output_dir)
    save_json(os.path.join(output_dir, "run_config.json"), {
        "run_id": RUN_ID,
        "variant": variant,
        "base_order_checkpoint": ORDER_BEST_CHECKPOINT_DIR,
        "train_rows": len(experiment_training_df),
        "valid_rows": len(experiment_validation_df),
        "max_train_steps": max_steps,
        "save_steps": save_steps,
        "estimated_steps_per_epoch": math.ceil(len(experiment_training_df) / GRADIENT_ACCUMULATION_STEPS),
        "note": "50 optimizer steps is intentionally a little more than one pass over 300 rows with grad_accum=8.",
        "order_output_with_spaces": ORDER_OUTPUT_WITH_SPACES,
        "learning_rate": LEARNING_RATE,
        "seed": SEED,
        "event_cache_path": ORDER_EVENT_CACHE_PATH,
        "frame_evidence_cache_path": ORDER_FRAME_EVIDENCE_CACHE_PATH,
    })
    del model
    gc.collect()
    torch.cuda.empty_cache()

if RUN_TRAINING:
    for variant in RUN_VARIANTS:
        train_variant(variant)
else:
    print("RUN_TRAINING is False; skipping.")


In [ ]:
# 9) Greedy eval A/D, choose winner, then winner 24-way
def checkpoint_path_for_variant(variant):
    output_dir = VARIANT_TO_OUTPUT_DIR[variant]
    preferred = os.path.join(output_dir, f"checkpoint-{MAX_TRAIN_STEPS}")
    if os.path.exists(os.path.join(preferred, "adapter_config.json")):
        return preferred
    if os.path.exists(os.path.join(output_dir, "adapter_config.json")):
        return output_dir
    checkpoints = sorted(glob.glob(os.path.join(output_dir, "checkpoint-*")))
    checkpoints = [path for path in checkpoints if os.path.exists(os.path.join(path, "adapter_config.json"))]
    if not checkpoints:
        raise RuntimeError(f"No checkpoint found for {variant}: {output_dir}")
    return checkpoints[-1]

def subgroup_metrics(pred_df, group_column, model_name):
    rows = []
    if group_column not in pred_df.columns:
        return rows
    for value, group in pred_df.groupby(group_column, dropna=False):
        metrics = compute_metrics(group.to_dict("records"))
        metrics.update({"model": model_name, "group_column": group_column, "group_value": value})
        rows.append(metrics)
    return rows

greedy_rows = []
subgroup_rows = []
greedy_prediction_paths = {}

if RUN_GREEDY_VARIANT_EVAL:
    for variant in RUN_VARIANTS:
        checkpoint_dir = checkpoint_path_for_variant(variant)
        model = load_order_adapter(checkpoint_dir, trainable=False)
        try:
            pred_path = os.path.join(EVAL_DIR, f"{variant.lower()}_greedy_predictions.csv")
            pred_df, metrics = evaluate_greedy(
                model,
                experiment_validation_df,
                variant,
                pred_path,
                limit_rows=GREEDY_EVAL_ROWS,
                desc=f"{variant} greedy",
            )
            metrics.update({
                "model": variant,
                "inference": "greedy",
                "checkpoint": checkpoint_dir,
                "eval_rows": GREEDY_EVAL_ROWS,
                "metric_group": "model_selection_100",
            })
            greedy_rows.append(metrics)
            greedy_prediction_paths[variant] = pred_path
            subgroup_rows.extend(subgroup_metrics(pred_df, "No_ordering", variant))
            subgroup_rows.extend(subgroup_metrics(pred_df, "ordering_type", variant))
        finally:
            del model
            gc.collect()
            torch.cuda.empty_cache()

greedy_metrics_df = pd.DataFrame(greedy_rows).sort_values(
    ["position_accuracy", "exact_match_accuracy", "valid_output_rate"],
    ascending=False,
).reset_index(drop=True)
greedy_metrics_df.to_csv(os.path.join(EVAL_DIR, "greedy_metrics.csv"), index=False)
subgroup_metrics_df = pd.DataFrame(subgroup_rows)
subgroup_metrics_df.to_csv(os.path.join(EVAL_DIR, "subgroup_metrics.csv"), index=False)
display(greedy_metrics_df)
display(subgroup_metrics_df)

winner_variant = greedy_metrics_df.iloc[0]["model"] if not greedy_metrics_df.empty else "A_CONTINUE"
winner_checkpoint = greedy_metrics_df.iloc[0]["checkpoint"] if not greedy_metrics_df.empty else checkpoint_path_for_variant(winner_variant)
print("winner:", winner_variant, winner_checkpoint)

final_rows = []
baseline_path = os.path.join(EVAL_DIR, "baseline_greedy_vs_24way_metrics.csv")
if os.path.exists(baseline_path):
    final_rows.extend(pd.read_csv(baseline_path).to_dict("records"))
final_rows.extend(greedy_rows)

winner_greedy_path = greedy_prediction_paths.get(winner_variant)
if winner_greedy_path and os.path.exists(winner_greedy_path):
    winner_greedy_30_df = pd.read_csv(winner_greedy_path).iloc[:CANDIDATE_SCORE_EVAL_ROWS].copy()
    winner_greedy_30_metrics = compute_metrics(winner_greedy_30_df.to_dict("records"))
    winner_greedy_30_metrics.update({
        "model": "Winner",
        "winner_variant": winner_variant,
        "inference": "greedy",
        "checkpoint": winner_checkpoint,
        "eval_rows": CANDIDATE_SCORE_EVAL_ROWS,
        "metric_group": "inference_comparison_30",
    })
    final_rows.append(winner_greedy_30_metrics)

if RUN_WINNER_24WAY_EVAL:
    model = load_order_adapter(winner_checkpoint, trainable=False)
    try:
        pred_df, metrics, candidate_df = evaluate_24way(
            model,
            experiment_validation_df,
            winner_variant,
            os.path.join(EVAL_DIR, f"{winner_variant.lower()}_winner"),
            limit_rows=CANDIDATE_SCORE_EVAL_ROWS,
            desc=f"{winner_variant} winner 24-way",
        )
        metrics.update({
            "model": "Winner",
            "winner_variant": winner_variant,
            "inference": "24-way",
            "checkpoint": winner_checkpoint,
            "eval_rows": CANDIDATE_SCORE_EVAL_ROWS,
            "metric_group": "inference_comparison_30",
        })
        final_rows.append(metrics)
    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()

final_comparison_df = pd.DataFrame(final_rows)
final_comparison_df.to_csv(os.path.join(EVAL_DIR, "final_comparison.csv"), index=False)
display(final_comparison_df)
print("saved run root:", RUN_ROOT)
